# Data

In [1]:
from pathlib import Path

from datasets import Dataset, DatasetDict, Value, ClassLabel, Features
from transformers import Trainer, TrainingArguments, RobertaTokenizer, RobertaForSequenceClassification

from tqdm.notebook import tqdm

In [2]:
PATH = Path("data/aclImdb")

In [54]:
def create_data(path):
    """
    Create a dataset from text files in a given directory.

    Args:
        path (Path): Path to the directory containing the dataset.

    Returns:
        Dataset: A Hugging Face Dataset object containing the text data and labels.
    """
    splits = ["train", "test"]
    labels = {"pos": 1, "neg": 0}
    data = {
        "train": {
            "text": [],
            "label": [],
        },
        "test": {
            "text": [],
            "label": [],
        }
    }

    for split in tqdm(splits, desc="Splits"):
        for label in tqdm(labels.keys(), desc="Labels", leave=False):
            for file in tqdm(
                iterable=(path / split / label).iterdir(),
                desc="Files",
                total=len(list((path / split / label).iterdir())),
                leave=False):
                if file.is_file():
                    with open(file, "r") as f:
                        for line in f:
                            line = line.strip()
                            if line:
                                data[split]["text"].append(line)
                                data[split]["label"].append(labels[label])
    return data

In [55]:
data = create_data(PATH)

Splits:   0%|          | 0/2 [00:00<?, ?it/s]

Labels:   0%|          | 0/2 [00:00<?, ?it/s]

Files:   0%|          | 0/12500 [00:00<?, ?it/s]

Files:   0%|          | 0/12500 [00:00<?, ?it/s]

Labels:   0%|          | 0/2 [00:00<?, ?it/s]

Files:   0%|          | 0/12500 [00:00<?, ?it/s]

Files:   0%|          | 0/12500 [00:00<?, ?it/s]

In [58]:
features = {
    "text": Value(dtype="string", id=None),
    "label": ClassLabel(num_classes=2, names=["neg", "pos"], names_file=None, id=None),
}

In [59]:
dataset_dict = DatasetDict({
    'train': Dataset.from_dict(data['train'], features=Features(features)),
    'test': Dataset.from_dict(data['test'],features=Features(features))
})

In [60]:
dataset_dict.save_to_disk("data/aclImdb_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/25000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/25000 [00:00<?, ? examples/s]

In [3]:
ds = DatasetDict.load_from_disk("data/aclImdb_dataset")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})


In [5]:
ds["train"].to_parquet("data/aclImdb_train.parquet")

Creating parquet from Arrow format:   0%|          | 0/25 [00:00<?, ?ba/s]

33432771

In [6]:
ds["test"].to_parquet("data/aclImdb_test.parquet")

Creating parquet from Arrow format:   0%|          | 0/25 [00:00<?, ?ba/s]

32650581

# Model

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
print(model)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [5]:
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

In [14]:
ds["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [15]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

In [16]:
encoded_dataset = ds.map(preprocess_function, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [11]:
tokenizer("Hello, this one sentence!", "And this sentence goes with it.")

{'input_ids': [0, 31414, 6, 42, 65, 3645, 328, 2, 2, 2409, 42, 3645, 1411, 19, 24, 4, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

# Sagemaker

In [1]:
import sagemaker
from sagemaker.huggingface import HuggingFace

/Users/A341705/Code/gda/.venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[03/20/25 12:14:00] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=568843;file:///Users/A341705/Code/gda/.venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=214606;file:///Users/A341705/Code/gda/.venv/lib/python3.12/site-packages/botocore/credentials.py#1352\1352]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/A341705/Library/Application Support/sagemaker/config.yaml
